In [1]:
import pyvista as pv
import gstools as gs
import numpy as np
from astropy.io import fits

In [2]:
#pv.set_jupyter_backend("trame")
pv.set_jupyter_backend("client")

In [3]:
N = 256
dim, spacing = (N, N, N), (1,1,1)
grid = pv.ImageData(dimensions=dim, spacing=spacing)

In [4]:
field_synth = 'velocity'
model_gs    = 'gaussian'
alpha       = 2.0
len_scale   = [16, 16, 16]
angles      = [0, 0, 0]
seed        = 19880127

In [5]:
model = gs.Gaussian(dim=3, len_scale=len_scale, angles=angles)
srf = gs.SRF(model, seed=seed)

In [6]:
field = srf.mesh

In [7]:
srf.mesh(grid, points='points', name='random-field')

array([-1.27964498, -1.23094961, -1.16044781, ...,  0.24949527,
        0.14480912,  0.04339519], shape=(16777216,))

In [8]:
#grid.contour(isosurfaces=6).plot(smooth_shading=True, use_transparency=True, opacity=0.3, cmap='coolwarm')

In [9]:
import os
import sys
import pyvista as pv

print("Process ID:", os.getpid())
print("Python:", sys.executable)
print("PyVista:", pv.__version__)
print("Backend:", pv.global_theme.jupyter_backend)

Process ID: 32292
Python: C:\Users\javas\anaconda3\envs\gstools\python.exe
PyVista: 0.48.4
Backend: client


In [10]:
print(
    "Extension available:",
    pv.global_theme.trame.jupyter_extension_available,
)
print(
    "Extension enabled:",
    pv.global_theme.trame.jupyter_extension_enabled,
)

Extension available: False
Extension enabled: False


In [11]:
print(type(srf.field))

<class 'numpy.ndarray'>


In [12]:
print(srf.field.shape)

(16777216,)


In [13]:
field_3d = np.asarray(grid.point_data['random-field']).reshape(dim, order='F')

In [14]:
print(type(field_3d))
print(field_3d.shape)
print(field_3d.min())
print(field_3d.max())
print(field_3d.mean())
print(field_3d.std())

<class 'numpy.ndarray'>
(256, 256, 256)
-3.5835032049882587
3.800860065441376
-0.09395132208732518
0.942278436429454


In [15]:
header = fits.Header()

header["MODEL"] = model_gs
header["SEED"]  = seed
header["M3D"]   = alpha

header["LSCALE_X"] = len_scale[0]
header["LSCALE_Y"] = len_scale[1]
header["LSCALE_Z"] = len_scale[2]

header["ANGLE_X"] = angles[0]
header["ANGLE_Y"] = angles[1]
header["ANGLE_Z"] = angles[2]

header["DX"] = spacing[0]
header["DY"] = spacing[1]
header["DZ"] = spacing[2]


In [16]:
name = model_gs + "_3d_" + field_synth 
print(name)

gaussian_3d_velocity


In [18]:
fits.writeto(
    name + ".fits",
    field_3d.astype("float32"),
    header=header,
    overwrite=True,
)